In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_curve, auc

plt.style.use("ggplot")

np.set_printoptions(suppress=True)

In [2]:
FILE_PATH = "biomet_data.csv"

raw_data = pd.read_csv(FILE_PATH, sep="\t", header=None)

print("Original Shape:", raw_data.shape)
display(raw_data.head())

Original Shape: (144, 1001)


,0,1,2,3,4,5,6,7,8,9,...,991,992,993,994,995,996,997,998,999,1000
0,307.02177,296.77448,213.03365,195.24468,229.44699,249.84867,293.53549,170.68021,276.23258,289.47556,...,82.094406,87.698309,80.645246,82.143254,80.269738,105.60992,143.89334,90.366463,133.56439,NaN
1,364.09315,348.58811,331.81183,289.80958,327.11110,327.28334,352.94458,373.86327,305.48485,309.39232,...,145.273050,122.472600,99.531238,107.465500,117.635370,141.09794,130.44142,114.079670,131.75557,NaN
2,239.10673,238.26485,289.61825,173.42357,281.41246,251.26869,267.15606,338.42246,232.21276,233.23617,...,126.828870,112.234890,104.602160,152.589500,130.194480,128.19904,180.96623,115.410410,216.64343,NaN
3,206.98782,241.55478,281.39093,219.13554,252.41240,241.61543,245.33423,290.04645,231.54125,223.89134,...,135.245790,127.371010,115.507560,112.364970,109.554370,144.03013,182.73806,124.573490,222.68632,NaN
4,192.14200,216.70194,245.03126,216.16553,220.47665,208.92883,209.83645,299.29639,201.96145,204.51265,...,142.308290,117.902140,113.258470,117.764660,137.740130,115.97421,145.43928,114.540650,125.31532,NaN


In [3]:
print("="*60)
print("DATASET VALIDATION")
print("="*60)

print(f"Rows    : {raw_data.shape[0]}")
print(f"Columns : {raw_data.shape[1]}")

print("\nMissing Values")
print(raw_data.isna().sum().sum())

print("\nCompletely Empty Columns")
print(raw_data.isna().all(axis=0).sum())

print("\nCompletely Empty Rows")
print(raw_data.isna().all(axis=1).sum())

DATASET VALIDATION
Rows    : 144
Columns : 1001

Missing Values
144

Completely Empty Columns
1

Completely Empty Rows
0


In [4]:
empty_cols = raw_data.columns[raw_data.isna().all()]

print("Empty Columns")

display(empty_cols)

Empty Columns


Index([1000], dtype='int64')

In [5]:
raw_data = raw_data.dropna(axis=1, how="all")

print("Shape after removing empty columns")

print(raw_data.shape)

Shape after removing empty columns
(144, 1000)


In [6]:
raw_data = raw_data.iloc[:,1:]

print(raw_data.shape)

(144, 999)


In [7]:
data = raw_data.T

data = data.apply(pd.to_numeric)

data.columns = [f"Feature_{i+1}" for i in range(data.shape[1])]

print(data.shape)

(999, 144)


In [8]:
print("="*60)
print("POST PROCESSING")
print("="*60)

print("Samples :", len(data))
print("Features:", data.shape[1])

print("NaNs :", data.isna().sum().sum())

POST PROCESSING
Samples : 999
Features: 144
NaNs : 0


I found that 1 column is empty and hence needed to do all above steps

In [10]:
# ============================================================
# ASSIGN USER IDs
# ============================================================

user_ids = []

for i in range(len(data)):
    user_ids.append(i // 10)

data["UserID"] = user_ids

print("Total Users :", data["UserID"].nunique())
print("Last User Samples :", (data["UserID"] == data["UserID"].max()).sum())

Total Users : 100
Last User Samples : 9


In [11]:
# ============================================================
# CREATE TRAIN / TEST SPLIT
# ============================================================

train_list = []
test_list = []

for user in sorted(data["UserID"].unique()):

    user_data = data[data["UserID"] == user]

    # First 5 samples -> Enrollment
    train_list.append(user_data.iloc[:5])

    # Remaining samples -> Testing
    test_list.append(user_data.iloc[5:])

train_data = pd.concat(train_list).reset_index(drop=True)
test_data = pd.concat(test_list).reset_index(drop=True)

print("Enrollment Samples :", len(train_data))
print("Testing Samples    :", len(test_data))

Enrollment Samples : 500
Testing Samples    : 499


In [12]:
# ============================================================
# VERIFY SPLIT
# ============================================================

print("=" * 60)

for user in [0, 1, 98, 99]:

    tr = len(train_data[train_data.UserID == user])
    te = len(test_data[test_data.UserID == user])

    print(
        f"User {user+1:3d} -> Train : {tr} | Test : {te}"
    )

User   1 -> Train : 5 | Test : 5
User   2 -> Train : 5 | Test : 5
User  99 -> Train : 5 | Test : 5
User 100 -> Train : 5 | Test : 4


In [13]:
feature_columns = [c for c in data.columns if c != "UserID"]

templates = (
    train_data
    .groupby("UserID")[feature_columns]
    .mean()
)

print("Template Matrix Shape :", templates.shape)

display(templates.head())

Template Matrix Shape : (100, 145)


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,Feature_10,...,Feature_136,Feature_137,Feature_138,Feature_139,Feature_140,Feature_141,Feature_142,Feature_143,Feature_144,SampleIndex
UserID,,,,,,,,,,,,,,,,,,,,,
0,236.869694,324.920792,246.797564,247.221816,221.460842,316.426444,372.058684,290.052658,305.699960,232.565022,...,265.697650,218.004900,177.579314,184.339326,153.546636,165.029512,175.031562,183.256830,178.583646,2.0
1,165.810614,200.499996,222.964600,196.365778,148.041894,122.221672,148.926620,163.263588,150.598826,143.886900,...,152.775398,125.286086,130.115906,122.254070,127.847866,120.944934,101.176942,111.732686,112.172841,12.0
2,190.165102,184.249054,183.790788,183.012400,222.184122,252.798596,211.997996,169.367262,152.707114,129.616127,...,179.261064,124.814924,130.232360,120.278174,113.928213,126.501332,120.927786,122.370367,126.648076,22.0
3,154.085062,196.222828,264.687136,378.080824,371.769052,208.044078,160.349370,139.480698,137.522952,117.853564,...,161.631798,133.890154,133.257172,131.451136,132.460256,133.277890,121.107906,99.902332,91.866722,32.0
4,184.940244,150.345246,140.123604,126.360356,143.671924,153.638444,120.276260,115.135450,115.417062,111.444204,...,157.951958,143.239656,124.295356,131.113798,109.967864,93.412053,101.247224,112.025034,109.489958,42.0


In [14]:
# ============================================================
# PREPARE MATRICES
# ============================================================

template_matrix = templates.values.astype(np.float64)

test_matrix = test_data[feature_columns].values.astype(np.float64)

test_labels = test_data["UserID"].values

print("Template Matrix :", template_matrix.shape)
print("Test Matrix     :", test_matrix.shape)
print("Labels          :", test_labels.shape)

Template Matrix : (100, 145)
Test Matrix     : (499, 145)
Labels          : (499,)


In [16]:
display(templates.head())

,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,Feature_10,...,Feature_136,Feature_137,Feature_138,Feature_139,Feature_140,Feature_141,Feature_142,Feature_143,Feature_144,SampleIndex
UserID,,,,,,,,,,,,,,,,,,,,,
0,236.869694,324.920792,246.797564,247.221816,221.460842,316.426444,372.058684,290.052658,305.699960,232.565022,...,265.697650,218.004900,177.579314,184.339326,153.546636,165.029512,175.031562,183.256830,178.583646,2.0
1,165.810614,200.499996,222.964600,196.365778,148.041894,122.221672,148.926620,163.263588,150.598826,143.886900,...,152.775398,125.286086,130.115906,122.254070,127.847866,120.944934,101.176942,111.732686,112.172841,12.0
2,190.165102,184.249054,183.790788,183.012400,222.184122,252.798596,211.997996,169.367262,152.707114,129.616127,...,179.261064,124.814924,130.232360,120.278174,113.928213,126.501332,120.927786,122.370367,126.648076,22.0
3,154.085062,196.222828,264.687136,378.080824,371.769052,208.044078,160.349370,139.480698,137.522952,117.853564,...,161.631798,133.890154,133.257172,131.451136,132.460256,133.277890,121.107906,99.902332,91.866722,32.0
4,184.940244,150.345246,140.123604,126.360356,143.671924,153.638444,120.276260,115.135450,115.417062,111.444204,...,157.951958,143.239656,124.295356,131.113798,109.967864,93.412053,101.247224,112.025034,109.489958,42.0


In [17]:
# ============================================================
# CREATE ENROLLMENT TEMPLATES
# ============================================================

feature_columns = [c for c in train_data.columns if c.startswith("Feature_")]

templates = (
    train_data
    .groupby("UserID")[feature_columns]
    .mean()
)

print("Template Matrix Shape:", templates.shape)
display(templates.head())

Template Matrix Shape: (100, 144)


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,Feature_10,...,Feature_135,Feature_136,Feature_137,Feature_138,Feature_139,Feature_140,Feature_141,Feature_142,Feature_143,Feature_144
UserID,,,,,,,,,,,,,,,,,,,,,
0,236.869694,324.920792,246.797564,247.221816,221.460842,316.426444,372.058684,290.052658,305.699960,232.565022,...,234.676246,265.697650,218.004900,177.579314,184.339326,153.546636,165.029512,175.031562,183.256830,178.583646
1,165.810614,200.499996,222.964600,196.365778,148.041894,122.221672,148.926620,163.263588,150.598826,143.886900,...,140.767502,152.775398,125.286086,130.115906,122.254070,127.847866,120.944934,101.176942,111.732686,112.172841
2,190.165102,184.249054,183.790788,183.012400,222.184122,252.798596,211.997996,169.367262,152.707114,129.616127,...,193.530754,179.261064,124.814924,130.232360,120.278174,113.928213,126.501332,120.927786,122.370367,126.648076
3,154.085062,196.222828,264.687136,378.080824,371.769052,208.044078,160.349370,139.480698,137.522952,117.853564,...,142.668638,161.631798,133.890154,133.257172,131.451136,132.460256,133.277890,121.107906,99.902332,91.866722
4,184.940244,150.345246,140.123604,126.360356,143.671924,153.638444,120.276260,115.135450,115.417062,111.444204,...,143.283922,157.951958,143.239656,124.295356,131.113798,109.967864,93.412053,101.247224,112.025034,109.489958


In [18]:
template_matrix = templates[feature_columns].to_numpy(dtype=np.float64)

test_matrix = test_data[feature_columns].to_numpy(dtype=np.float64)

test_labels = test_data["UserID"].to_numpy()

print(template_matrix.shape)
print(test_matrix.shape)

(100, 144)
(499, 144)
